# Estimate single cell parameters

Computes the per-species block of `codeconv_config.json` from a folder of 10x scRNA-seq datasets.

Outputs the following fields for the selected species:
- `hk_profiles`: median across datasets of the per-gene mean log1p(CP10K) for each housekeeping gene.
- `engine_parameters`: median library `mu` (mean UMI per cell) and `phi` (overdispersion = var/mean^2) across datasets.
- `noise_regex` and `qc_markers`: copied from defaults (or preserved from the existing config if present).

Other species' blocks in the config are left untouched. The top-level `min_topic_percentage` is preserved (or initialized to 0.05 if missing).

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import scipy.io
import gzip
from tqdm import tqdm

warnings.filterwarnings('ignore')

In [ ]:
config_path   = 'codeconv_config.json'
species       = 'hs'   # 'hs' | 'mm' | 'other'
datasets_root = '.'    # folder containing one subfolder per scRNA-seq dataset
min_avg_lib_size = 100 # skip datasets whose mean lib size is below this

In [ ]:
# Defaults seeded into the config when no prior species block exists.
# If the config already has a species block, its HK gene list, qc_markers, and
# noise_regex are preserved and only the numerical values are recomputed.

SPECIES_DEFAULTS = {
    'hs': {
        'noise_regex': r'^MT-|^RP[SL][0-9]+|^LINC|^MIR|^AC[0-9]+',
        'hk_genes': [
            'RPL19', 'TUBB', 'EEF1G', 'PPIA', 'GAPDH', 'ABCF1', 'SDHA', 'OAZ1',
            'G6PD', 'ALAS1', 'GUSB', 'HPRT1', 'POLR2A', 'POLR1B', 'TBP'
        ],
        'qc_markers': [
            'CD3E', 'CD4', 'CD8A', 'MS4A1', 'CD19', 'PTPRC',
            'HBB', 'CD14', 'FCGR3A', 'CD34', 'NCAM1', 'JCHAIN',
            'EPCAM', 'KRT18', 'COL1A1', 'DCN', 'PECAM1', 'ERBB2'
        ],
    },
    'mm': {
        'noise_regex': r'^mt-|^Rp[sl][0-9]+|^Gm[0-9]+|^Mir|^Rik$',
        'hk_genes': [
            'Rpl19', 'Tubb5', 'Eef1g', 'Ppia', 'Gapdh', 'Abcf1', 'Sdha', 'Oaz1',
            'G6pdx', 'Alas1', 'Gusb', 'Hprt', 'Polr2a', 'Polr1b', 'Tbp'
        ],
        'qc_markers': [
            'Cd3e', 'Cd4', 'Cd8a', 'Ms4a1', 'Cd19', 'Ptprc',
            'Hbb-bs', 'Cd14', 'Fcgr3', 'Cd34', 'Ncam1', 'Jchain',
            'Epcam', 'Krt18', 'Col1a1', 'Dcn', 'Pecam1', 'Erbb2'
        ],
    },
    'other': {
        'noise_regex': '',
        'hk_genes': [],
        'qc_markers': [],
    },
}

In [ ]:
def load_10x_data(path):
    """Load a 10x cell x gene matrix and gene symbols from a directory.

    Supports CellRanger 3+ ('features.tsv.gz') and CellRanger 2 ('genes.tsv.gz').
    Returns (csr_matrix [genes x cells], gene_symbols) or (None, None) on failure.
    """
    matrix_path = os.path.join(path, 'matrix.mtx.gz')
    features_path = os.path.join(path, 'features.tsv.gz')
    if not os.path.exists(features_path):
        features_path = os.path.join(path, 'genes.tsv.gz')
    if not (os.path.exists(matrix_path) and os.path.exists(features_path)):
        return None, None
    try:
        features = pd.read_csv(features_path, sep='\t', header=None, compression='gzip')
        gene_symbols = features[1].values.astype(str)
        with gzip.open(matrix_path, 'rb') as f:
            mat = scipy.io.mmread(f)
        return mat.tocsr(), gene_symbols
    except Exception:
        return None, None

In [ ]:
def load_existing_config(config_path):
    """Load existing config or return a minimal scaffold."""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            return json.load(f)
    return {'min_topic_percentage': 0.05, 'species_profiles': {}}


def resolve_species_inputs(config, species):
    """Decide HK gene list, qc_markers, and noise_regex.

    If the config already has a block for this species, preserve its gene lists and
    regex. Otherwise seed from SPECIES_DEFAULTS.
    """
    profiles = config.get('species_profiles', {})
    if species in profiles and profiles[species].get('hk_profiles'):
        block = profiles[species]
        hk_genes = list(block['hk_profiles'].keys())
        qc_markers = list(block.get('qc_markers', []))
        noise_regex = block.get('noise_regex', '')
        print(f"  preserving existing config block for '{species}' ({len(hk_genes)} HK genes)")
    else:
        defaults = SPECIES_DEFAULTS[species]
        hk_genes = list(defaults['hk_genes'])
        qc_markers = list(defaults['qc_markers'])
        noise_regex = defaults['noise_regex']
        print(f"  seeding from SPECIES_DEFAULTS for '{species}' ({len(hk_genes)} HK genes)")
    return hk_genes, qc_markers, noise_regex

In [ ]:
def estimate_species_block(species, datasets_root, hk_genes, qc_markers, noise_regex,
                           min_avg_lib_size=100):
    """Walk dataset folders and accumulate per-gene log1p(CP10K) plus library stats.

    Returns the species_profiles[species] block ready for codeconv_config.json.
    """
    dataset_dirs = [
        d for d in os.listdir(datasets_root)
        if os.path.isdir(os.path.join(datasets_root, d)) and not d.startswith('.')
    ]
    print(f'Scanning {len(dataset_dirs)} dataset folders under {datasets_root!r}...')

    engine_stats = []
    hk_log_cp10k = {g: [] for g in hk_genes}

    # Case-insensitive matching for robustness, but report actual gene name as found in data
    hk_lookup = {g.lower(): g for g in hk_genes}

    for dataset_id in tqdm(dataset_dirs):
        path = os.path.join(datasets_root, dataset_id)
        mat, symbols = load_10x_data(path)
        if mat is None:
            continue

        lib_sizes = np.array(mat.sum(axis=0)).flatten()
        mu_s = float(np.mean(lib_sizes))
        if mu_s < min_avg_lib_size:
            continue
        phi_s = float(np.var(lib_sizes) / (mu_s ** 2)) if mu_s > 0 else 0.0
        engine_stats.append({'mu': mu_s, 'phi': phi_s})

        # Per-gene log1p(CP10K) means for the HK panel
        for i, sym in enumerate(symbols):
            target = hk_lookup.get(sym.lower())
            if target is None:
                continue
            vec = np.array(mat[i, :].toarray()).flatten()
            cp10k = (vec / (lib_sizes + 1e-9)) * 10000
            log1p_cp10k = np.log1p(cp10k)
            hk_log_cp10k[target].append(float(np.mean(log1p_cp10k)))

    df_engine = pd.DataFrame(engine_stats)
    if df_engine.empty:
        raise RuntimeError('No usable datasets found; check datasets_root and min_avg_lib_size.')

    final_engine = {
        'mu': float(df_engine['mu'].median()),
        'phi': float(df_engine['phi'].median()),
    }
    final_hk = {
        g: float(np.median(vals))
        for g, vals in hk_log_cp10k.items()
        if len(vals) > 0
    }

    missing = [g for g in hk_genes if g not in final_hk]
    if missing:
        print(f"  warning: {len(missing)} HK genes never observed in any dataset: {missing}")

    return {
        'noise_regex': noise_regex,
        'hk_profiles': final_hk,
        'qc_markers': qc_markers,
        'engine_parameters': final_engine,
    }

In [ ]:
config = load_existing_config(config_path)
hk_genes, qc_markers, noise_regex = resolve_species_inputs(config, species)

block = estimate_species_block(
    species, datasets_root,
    hk_genes=hk_genes,
    qc_markers=qc_markers,
    noise_regex=noise_regex,
    min_avg_lib_size=min_avg_lib_size,
)

print()
print(f"engine_parameters: mu={block['engine_parameters']['mu']:.2f}   phi={block['engine_parameters']['phi']:.4f}")
print(f"hk_profiles: {len(block['hk_profiles'])}/{len(hk_genes)} genes calibrated")
for g, v in block['hk_profiles'].items():
    print(f"  {g:>10s}  {v:.4f}")

In [ ]:
# Merge into the config, preserving other species' blocks and top-level min_topic_percentage
config.setdefault('min_topic_percentage', 0.05)
config.setdefault('species_profiles', {})
config['species_profiles'][species] = block

with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

print(f'Wrote {config_path}: species_profiles[{species!r}] updated.')
print(f"Other species in config: {sorted(s for s in config['species_profiles'] if s != species)}")